# WYR Data Analysis
Categorises model-generated question pairings by theme using a sentence-embedding model, then visualises category distributions and moderation (upvote / downvote / flag) scores.

## 1 · Imports & Setup

In [1]:
from sentence_transformers import SentenceTransformer
import hdbscan
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import sqlite3
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime
import matplotlib.colors as mcolors
import seaborn

TIME = datetime.today().strftime("%Y-%m-%d")
print(f"Run date: {TIME}")

Run date: 2026-06-26


## 2 · Embedding Model & Category Definitions

In [2]:
ST_MODEL = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Sentence-transformer loaded.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Sentence-transformer loaded.


In [3]:
CATEGORIES = {
    "animals":       ["animals", "pets", "creatures", "wildlife"],
    "body parts":    ["body parts", "limbs", "organs", "physical features", "arms", "legs"],
    "money":         ["money", "wealth", "finances", "$", "rich", "dollars", "currency"],
    "food / drink":  ["food", "drink", "eating", "meals", "cooking", "drinking"],
    "life":          ["life", "survival", "status", "well-being", "death", "dying",
                      "mortality", "afterlife", "living forever"],
    "ability":       ["superpower", "ability", "special ability", "magic power",
                      "power", "skill", "be able to"],
    "curse":         ["curse", "situation", "stuck", "trapped", "bewitched",
                      "inhibited", "rules", "forced"],
    "age / time":    ["age", "time", "youth", "old age", "going back in time"],
    "people":        ["people", "population", "persons", "everybody", "strangers",
                      "neighbors", "friends", "occupation", "job", "career"],
    "locations":     ["location", "countries", "cities", "towns", "world", "space"],
    "sentience":     ["become", "becoming", "inanimate", "transform", "stuck",
                      "mutation", "live as inanimate object", "be a", "be an"],
    "miscellaneous": ["miscellaneous", "other", "general", "random", "unrelated"],
}

MISC_THRESH = 0.50
print(f"{len(CATEGORIES)} categories defined.")

12 categories defined.


In [4]:
print("Pre-computing category embeddings...")
category_embeddings = {
    cat: ST_MODEL.encode(phrases)
    for cat, phrases in CATEGORIES.items()
}
print("Done.")

Pre-computing category embeddings...
Done.


## 3 · Helper Functions

In [5]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
def categorize_pairing(pairing):
    """Assign a main and secondary category to an option-A / option-B pairing."""
    combined        = f"{pairing['a']} or {pairing['b']}"
    query_embedding = ST_MODEL.encode(combined)

    best_category    = None
    best_score       = -1
    closest_category = "n/a"
    second_score     = 0

    for category, phrase_embeddings in category_embeddings.items():
        scores = [cosine_similarity(query_embedding, pe) for pe in phrase_embeddings]
        score  = max(scores)
        if score > best_score:
            closest_category = best_category
            second_score     = best_score
            best_score       = score
            best_category    = category
        elif second_score < score:
            closest_category = category
            second_score     = score

    if best_score < MISC_THRESH:
        closest_category = best_category
        best_category    = "miscellaneous"

    return {
        **pairing,
        "main_category":      best_category,
        "secondary_category": closest_category,
        "score":              round(float(best_score), 4),
    }

## 4 · Load Data from Database

In [7]:
conn = sqlite3.connect('../wyr_votes.db')
df   = pd.read_sql_query("SELECT model_name, option_a, option_b FROM model_outputs", conn)
conn.close()

# normalise llama -> tinyllama
df['model_name'] = df['model_name'].apply(lambda x: "tinyllama" if x == "llama" else x)

ALL_MODELS = sorted(m for m in df['model_name'].unique() if m != "deepseek")

print(f"Loaded {len(df):,} rows from model_outputs.")
print(f"Models: {ALL_MODELS}")
df.tail()

Loaded 4,686 rows from model_outputs.
Models: ['gemma', 'meta-llama', 'qwen', 'smol', 'tencent', 'tinyllama']


,model_name,option_a,option_b
4681,qwen,10 years of being a total asshole,One good day
4682,smol,Would you rather...\nA) Have to say the first ...,wrong). \n B) Can only answer one of these two...
4683,gemma,"Be able to talk for 10 minutes, but only when ...",Only be able to dance like a frog when it rains
4684,tencent,A 1000m square,A 1.5km square
4685,tinyllama,Be 50 in a tube,be the last person standing when everyones stu...


In [8]:
MODEL_SET   = {}
num_entries = 0

for _, entry in df.iterrows():
    if entry['option_a'] == "Option A" or entry['option_b'] == "Option B":
        continue
    model = entry['model_name']
    if model not in ALL_MODELS:
        continue
    MODEL_SET.setdefault(model, []).append(entry)
    num_entries += 1

for m, entries in MODEL_SET.items():
    print(f"  {m}: {len(entries):,} entries")
print(f"Total: {num_entries:,}")

  tinyllama: 851 entries
  smol: 851 entries
  qwen: 851 entries
  gemma: 850 entries
  meta-llama: 642 entries
  tencent: 640 entries
Total: 4,685


## 5 · Categorise All Pairings
Embeds every question pair and assigns a main + secondary category.  
Takes a few minutes on first run depending on hardware.

In [10]:
with open("og_dataset.json", "r") as f:
    og_data = json.load(f)

if isinstance(og_data, dict) and "data" in og_data:
    og_data = og_data["data"]

def normalize_record(record):
    if not isinstance(record, dict):
        raise ValueError("Expected each record to be a dict")

    if "a" in record and "b" in record:
        return {"a": record["a"], "b": record["b"]}
    if "option_a" in record and "option_b" in record:
        return {"a": record["option_a"], "b": record["option_b"]}
    if "optionA" in record and "optionB" in record:
        return {"a": record["optionA"], "b": record["optionB"]}

    raise ValueError(f"Unrecognized record format: {list(record.keys())}")

OG_CAT_PAIRS = [
    categorize_pairing(normalize_record(record))
    for record in og_data
]

print(f"Loaded {len(og_data):,} records from og_dataset.json")
print(f"Categorised {len(OG_CAT_PAIRS):,} original dataset pairings")
OG_CAT_PAIRS[:3]

Loaded 314 records from og_dataset.json
Categorised 314 original dataset pairings


[{'a': 'always have your room be the perfect temperature when going to sleep',
  'b': 'be able to immediately find the perfect sleeping position',
  'main_category': 'ability',
  'secondary_category': 'miscellaneous',
  'score': 0.6202},
 {'a': 'be bound to the same spot everyday for 5 hours at a time (body not and you are standing upward NOT comfortable) but for every word you speak during that time you get a dollar',
  'b': 'be a sentient weasel being exploited by a circus as a freak animal who can understand english',
  'main_category': 'curse',
  'secondary_category': 'sentience',
  'score': 0.5678},
 {'a': "everytime somebody around you said the magic word (but the radius of 'around' is like 100 km) you get given 12 donuts",
  'b': 'just have a permanent casual job as a grave representer so you vouch for dead people whenever you want to get prettier graves',
  'main_category': 'life',
  'secondary_category': 'people',
  'score': 0.5186}]

In [9]:
CAT_PAIRS = {}
with tqdm(total=num_entries, desc="Categorising") as pbar:
    for model, entries in MODEL_SET.items():
        CAT_PAIRS[model] = []
        for e in entries:
            CAT_PAIRS[model].append(
                categorize_pairing({'a': e['option_a'], 'b': e['option_b']})
            )
            pbar.update(1)

print("Categorisation complete.")

Categorising:   0%|          | 0/4685 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6 · Per-Model Category Charts
For each model: a vertical stacked bar showing **main** (blue) vs **secondary** (green) category counts.

In [ ]:
def vis_data_cat(data, model_name):
    main_cts   = {m: 0 for m in CATEGORIES}
    second_cts = {m: 0 for m in CATEGORIES}

    for entry in data:
        main_cts[entry['main_category']]        += 1
        second_cts[entry['secondary_category']] += 1

    blue_values  = [main_cts[m]   for m in CATEGORIES]
    green_values = [second_cts[m] for m in CATEGORIES]

    fig, ax = plt.subplots(figsize=(14, 6))
    x_pos   = range(len(CATEGORIES))
    ax.bar(x_pos, blue_values,  label='Main',      color='blue',  alpha=0.8)
    ax.bar(x_pos, green_values, label='Secondary', color='green', alpha=0.8,
           bottom=blue_values)

    ax.set_xlabel('Category',          fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Entries', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name} MODEL THEME (Stacked Bar Chart) — {len(data)}',
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(CATEGORIES, rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'graphs/{model_name}-[{TIME}].png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for model, dat in CAT_PAIRS.items():
    vis_data_cat(dat, model)

## 7 · Stacked Bar — All Models by Category
Horizontal chart: **Y-axis** = categories, **X-axis** = question count, each model colour-coded.

In [ ]:
def vis_stacked_all_models(cat_pairs):
    categories    = list(CATEGORIES.keys())
    models        = sorted(cat_pairs.keys())
    palette       = plt.cm.tab10.colors
    model_colours = {m: palette[i % len(palette)] for i, m in enumerate(models)}

    counts = {m: {c: 0 for c in categories} for m in models}
    for model, entries in cat_pairs.items():
        for e in entries:
            counts[model][e["main_category"]] += 1

    fig, ax = plt.subplots(figsize=(10, max(5, len(categories) * 0.55)))
    lefts   = [0] * len(categories)

    for model in models:
        values = [counts[model][c] for c in categories]
        bars   = ax.barh(
            categories, values, left=lefts,
            label=model, color=model_colours[model],
            edgecolor="white", height=0.7,
        )
        for bar_patch, val in zip(bars, values):
            if val > 0:
                ax.text(
                    bar_patch.get_x() + bar_patch.get_width() / 2,
                    bar_patch.get_y() + bar_patch.get_height() / 2,
                    str(val), ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold",
                )
        lefts = [l + v for l, v in zip(lefts, values)]

    ax.invert_yaxis()
    ax.set_xlabel("Number of questions", fontsize=11)
    ax.set_ylabel("Category",            fontsize=11)
    ax.set_title("Questions per category by model", fontsize=13, fontweight="bold")
    ax.legend(title="Model", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    plt.savefig(f'graphs/STACKED_BY_MODEL-{TIME}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
vis_stacked_all_models(CAT_PAIRS)

## 8 · Load Moderation Data
Reads upvoted, downvoted, and flagged questions from the JSON files produced by `proc_moderate_data.py`.

In [ ]:
with open("upvote_questions.json", 'r') as f:
    upvote_dat = json.load(f)

with open("bad_questions.json", 'r') as f:
    bad_dat      = json.load(f)
    downvote_dat = bad_dat['downvotes']
    flagged_dat  = bad_dat['flagged']

print(f"Upvotes : {len(upvote_dat):,}")
print(f"Downvotes: {len(downvote_dat):,}")
print(f"Flagged : {len(flagged_dat):,}")

## 9 · Categorise Moderation Data

In [ ]:
def categorize_mod_list(records):
    result = []
    for u in records:
        c = categorize_pairing({'a': u['optionA'], 'b': u['optionB']})
        c['model'] = "tinyllama" if u['MODEL'] == "llama" else u['MODEL']
        result.append(c)
    return result

upvote_cats   = categorize_mod_list(upvote_dat)
downvote_cats = categorize_mod_list(downvote_dat)
flagged_cats  = categorize_mod_list(flagged_dat)

print("Moderation data categorised.")

## 10 · Vote-Score Heatmaps
Score per `(category, model)` cell:  
`score = +3 × upvotes  −1 × downvotes  −3 × flags`  
Green = positive reception, red = negative.

In [ ]:
def vote_score_dataframe(ups, downs, flags):
    y_axis   = sorted(CATEGORIES.keys())
    x_axis   = sorted(ALL_MODELS)
    arr_main = np.zeros((len(y_axis), len(x_axis)))
    arr_sec  = np.zeros((len(y_axis), len(x_axis)))

    for records, score in [(ups, 3), (downs, -1), (flags, -3)]:
        for u in records:
            model = u['model']
            if model not in x_axis:
                continue
            mi = x_axis.index(model)
            arr_main[y_axis.index(u['main_category'])][mi]      += score
            arr_sec [y_axis.index(u['secondary_category'])][mi] += score

    return (
        pd.DataFrame(arr_main, index=y_axis, columns=x_axis),
        pd.DataFrame(arr_sec,  index=y_axis, columns=x_axis),
    )

In [ ]:
main_df, sec_df = vote_score_dataframe(upvote_cats, downvote_cats, flagged_cats)
main_df

In [ ]:
def showHeatmap(m, title):
    stoplight = seaborn.color_palette("blend:#ff0000,#fff,#00ff00", as_cmap=True)
    fig, ax   = plt.subplots(figsize=(12, 8))
    seaborn.heatmap(
        m, annot=True, cmap=stoplight, center=0,
        vmin=m.min().min(), vmax=m.max().max(), ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("MODELS")
    ax.set_ylabel("CATEGORIES")
    fig.tight_layout()
    fig.savefig(f"graphs/HEATMAP-{title}-{TIME}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
showHeatmap(main_df, "MAIN CATEGORIES")
showHeatmap(sec_df,  "SECONDARY CATEGORIES")